In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/q21_rewrite/checkpoints/pre_cell_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 4 ###

# Filter lineitems where receipt date > commit date
mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
lineitem_date = lineitem[mask_date][['L_ORDERKEY', 'L_SUPPKEY']]

# Compute unique supplier counts per order before and after the date filter
o_orig = (
    lineitem[['L_ORDERKEY', 'L_SUPPKEY']]
    .drop_duplicates()
    .groupby('L_ORDERKEY', as_index=False, sort=False)['L_SUPPKEY']
    .nunique()
    .rename(columns={'L_SUPPKEY': 'orig_count'})
)
o_after = (
    lineitem_date.drop_duplicates()
    .groupby('L_ORDERKEY', as_index=False, sort=False)['L_SUPPKEY']
    .nunique()
    .rename(columns={'L_SUPPKEY': 'after_count'})
)

# Identify orders with >1 supplier originally and exactly 1 after the date filter
valid_orders = (
    o_orig[o_orig.orig_count > 1]
    .merge(o_after[o_after.after_count == 1], on='L_ORDERKEY', how='inner')[['L_ORDERKEY']]
)

# Further restrict to orders with status 'F'
orders_f = orders[orders.O_ORDERSTATUS == 'F'][['O_ORDERKEY']]
valid_orders = (
    valid_orders
    .merge(orders_f, left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')[['L_ORDERKEY']]
)

# Filter the date-qualified lineitems to only those valid orders
li_final = lineitem_date[lineitem_date.L_ORDERKEY.isin(valid_orders.L_ORDERKEY)][['L_SUPPKEY']]

# Join with supplier and nation, then aggregate and sort
supplier_sel = supplier[['S_SUPPKEY', 'S_NATIONKEY', 'S_NAME']]
nation_sa = nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']]

total = (
    li_final
    .merge(supplier_sel, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')[['S_NATIONKEY', 'S_NAME']]
    .merge(nation_sa, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')[['S_NAME']]
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size': 'NUMWAIT'})
    .sort_values(by=['NUMWAIT', 'S_NAME'], ascending=[False, True])
)

CPU times: user 165 ms, sys: 98.2 ms, total: 264 ms
Wall time: 277 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q21/rewritten/o4_mini_high_small/checkpoints/post_cell_4_try_3.pickle